<a href="https://colab.research.google.com/github/TheRenster/Projects_Demos/blob/main/Job_Alignment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --force-reinstall PyPDF2
!pip install python-docx

  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)
  Attempting uninstall: PyPDF2
    Found existing installation: PyPDF2 3.0.1
    Uninstalling PyPDF2-3.0.1:
      Successfully uninstalled PyPDF2-3.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 21.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import io
from google.colab import files
import csv
import re

# Import PyPDF2 and python-docx with error handling
try:
    import PyPDF2
except ImportError:
    print("PyPDF2 not installed. Please install it using: !pip install PyPDF2")
    PyPDF2 = None

try:
    from docx import Document
except ImportError:
    print("python-docx not installed. Please install it using: !pip install python-docx")
    Document = None

# Load BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Function to extract text from PDF resume
def extract_text_from_pdf(pdf_content):
    if PyPDF2 is None:
        print("Cannot process PDF: PyPDF2 is not available.")
        return ''
    try:
        reader = PyPDF2.PdfReader(io.BytesIO(pdf_content))
        text = ''
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + ' '
        return text
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return ''

# Function to extract text from DOCX resume
def extract_text_from_docx(docx_content):
    if Document is None:
        print("Cannot process DOCX: python-docx is not available.")
        return ''
    try:
        doc = Document(io.BytesIO(docx_content))
        text = ''
        for para in doc.paragraphs:
            text += para.text + '\n'
        return text
    except Exception as e:
        print(f"Error reading DOCX: {e}")
        return ''

# Function to clean text
def clean_text(text):
    if not text or not isinstance(text, str):
        return ''
    text = re.sub(r'\s+', ' ', text)  # Remove extra whitespace
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    return text.lower().strip()

# Function to format skills for readability
def format_skills(skills):
    if not skills:
        return []
    # Split skills by semicolon or comma, handling concatenated strings
    skill_list = re.split(r'[;,]', skills)
    # Clean and capitalize each skill
    formatted = []
    for skill in skill_list:
        skill = skill.strip().lower()
        if skill:
            # Map common skills to proper formatting
            skill_map = {
                'c': 'C', 'cc': 'C++', 'cpp': 'C++', 'csharp': 'C#',
                'javascript': 'JavaScript', 'typescript': 'TypeScript',
                'htmlcss': 'HTML/CSS', 'html': 'HTML', 'css': 'CSS',
                'node': 'Node.js', 'nodejs': 'Node.js', 'reactjs': 'React.js',
                'vuejs': 'Vue.js', 'ruby, rails': 'Ruby on Rails', 'ruby on rails': 'Ruby on Rails',
                'bashshell': 'Bash/Shell', 'sql': 'SQL', 'mysql': 'MySQL',
                'postgresql': 'PostgreSQL', 'aws': 'AWS', 'git': 'Git',
                'jquery': 'jQuery', 'laravel': 'Laravel', 'express': 'Express',
                'java': 'Java', 'perl': 'Perl', 'ruby': 'Ruby', 'php': 'PHP',
                'python': 'Python', 'cpython': 'Python'  # Handle typo from output
            }
            formatted.append(skill_map.get(skill, skill.capitalize()))
    return formatted[:5]  # Limit to 5 skills for brevity

# Function to get BERT embeddings
def get_bert_embeddings(text):
    if not text:
        return np.zeros(768)  # Return zero vector if text is empty
    inputs = tokenizer(text, return_tensors='pt', max_length=512, truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embeddings

# Function to calculate similarity between resume and job skills
def calculate_similarity(resume_embedding, job_embedding):
    if np.all(resume_embedding == 0) or np.all(job_embedding == 0):
        return 0.0
    return cosine_similarity([resume_embedding], [job_embedding])[0][0]

# Function to generate recommendations
def generate_recommendations(resume_text, job_skills, similarity_score):
    recommendations = []
    if not resume_text or not job_skills:
        recommendations.append("Unable to analyze due to missing or invalid resume/job skills.")
        return recommendations

    resume_words = set(clean_text(resume_text).split())
    job_words = set(clean_text(job_skills).split())
    missing_keywords = job_words - resume_words
    formatted_keywords = format_skills(','.join(missing_keywords))

    if similarity_score < 0.6:
        recommendations.append("Your resume has low alignment with the job skills (Likelihood: <60%).")
        if formatted_keywords:
            recommendations.append("Consider adding these skills/technologies:")
            for keyword in formatted_keywords:
                recommendations.append(f"  - {keyword}")
        recommendations.append("Highlight relevant technical skills and projects matching job requirements.")
        recommendations.append("Consider tailoring your resume to include more of the listed technologies.")
    elif similarity_score < 0.8:
        recommendations.append("Your resume is moderately aligned (Likelihood: 60-80%). To improve:")
        if formatted_keywords:
            recommendations.append("Consider adding these skills/technologies:")
            for keyword in formatted_keywords:
                recommendations.append(f"  - {keyword}")
        recommendations.append("Emphasize projects or experiences using the job’s key technologies.")
        recommendations.append("Ensure your technical skills section is prominent and relevant.")
    else:
        recommendations.append("Your resume is well-aligned with the job skills (Likelihood: >80%).")
        recommendations.append("Polish formatting and check for typos to stand out.")
        recommendations.append("Consider adding quantifiable achievements with these technologies.")
    return recommendations

# Function to upload resume in Colab
def upload_resume():
    print("Please upload your resume (PDF or DOCX)...")
    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded.")
        return ''

    for filename, content in uploaded.items():
        if filename.endswith('.pdf'):
            return extract_text_from_pdf(content)
        elif filename.endswith('.docx'):
            return extract_text_from_docx(content)
    print("Unsupported file format. Please upload a PDF or DOCX file.")
    return ''

# Function to load dataset with error handling
def load_dataset(dataset_path):
    encodings = ['utf-8', 'latin1', 'iso-8859-1']
    for encoding in encodings:
        try:
            df = pd.read_csv(
                dataset_path,
                encoding=encoding,
                quoting=csv.QUOTE_ALL,
                on_bad_lines='skip',  # Skip malformed rows
                low_memory=False
            )
            print(f"Successfully loaded dataset with {encoding} encoding. Rows: {len(df)}")
            return df
        except Exception as e:
            print(f"Failed with {encoding} encoding: {e}")

    # Fallback: Try loading a subset
    try:
        print("Attempting to load subset of dataset...")
        df = pd.read_csv(
            dataset_path,
            encoding='utf-8',
            quoting=csv.QUOTE_ALL,
            on_bad_lines='skip',
            nrows=50000  # Load first 50,000 rows
        )
        print(f"Loaded subset of dataset (first 50,000 rows). Rows: {len(df)}")
        return df
    except Exception as e:
        print(f"Failed to load subset: {e}")
        return None

# Main function to analyze resume
def analyze_resume(dataset_path='stackoverflow_full.csv'):
    # Load dataset
    df = load_dataset(dataset_path)
    if df is None:
        print("Could not load dataset. Please check file format and content.")
        return

    required_columns = ['HaveWorkedWith']
    available_columns = df.columns.tolist()
    if not all(col in available_columns for col in required_columns):
        print(f"Error: Dataset must contain {required_columns}. Found: {available_columns}")
        return
    job_skills = df['HaveWorkedWith'].astype(str).tolist()
    countries = df.get('Country', pd.Series(['Unknown'] * len(df))).astype(str).tolist()
    salaries = df.get('PreviousSalary', pd.Series([0.0] * len(df))).astype(float).tolist()

    # Upload resume
    resume_text = upload_resume()
    if not resume_text:
        print("No resume uploaded or invalid file.")
        return

    # Clean and process resume
    resume_text = clean_text(resume_text)
    if not resume_text:
        print("No valid text extracted from resume.")
        return
    resume_embedding = get_bert_embeddings(resume_text)

    # Analyze against each job's skills (limit to first 100 for speed)
    results = []
    for i, (skills, country, salary) in enumerate(zip(job_skills[:100], countries[:100], salaries[:100])):
        if not skills or pd.isna(skills):
            continue
        skills = clean_text(skills)
        if not skills:
            continue
        job_embedding = get_bert_embeddings(skills)
        similarity = calculate_similarity(resume_embedding, job_embedding)
        likelihood = min(similarity * 100, 100)  # Convert to percentage, cap at 100
        recommendations = generate_recommendations(resume_text, skills, similarity)

        # Store results
        results.append({
            'job_id': i + 1,
            'country': country,
            'salary': salary,
            'alignment_score': similarity,
            'likelihood': likelihood,
            'skills': df['HaveWorkedWith'].iloc[i][:100] + '...',
            'recommendations': recommendations
        })

    # Sort results by likelihood (descending) then alignment score (descending)
    results = sorted(results, key=lambda x: (x['likelihood'], x['alignment_score']), reverse=True)

    # Print sorted results
    print("\nResume Analysis Results (Top 100 Jobs, Sorted by Likelihood):")
    print("-" * 50)
    for result in results:
        print(f"\nJob #{result['job_id']} (Country: {result['country']}, Salary: ${result['salary']:,.2f})")
        print(f"Alignment Score: {result['alignment_score']:.2f}")
        print(f"Likelihood of Getting Job: {result['likelihood']:.1f}%")
        print(f"Skills: {result['skills']}")
        print("Recommendations:")
        for rec in result['recommendations']:
            print(f"- {rec}")
        print("-" * 50)

# Run the analyzer
if __name__ == "__main__":
    analyze_resume()

Successfully loaded dataset with utf-8 encoding. Rows: 73462
Please upload your resume (PDF or DOCX)...


Saving KKELLEY_CURRENTRESUME.pdf to KKELLEY_CURRENTRESUME.pdf

Resume Analysis Results (Top 100 Jobs, Sorted by Likelihood):
--------------------------------------------------

Job #27 (Country: Poland, Salary: $45,564.00)
Alignment Score: 0.65
Likelihood of Getting Job: 65.1%
Skills: Bash/Shell;C#;Dart;Delphi;Go;HTML/CSS;Java;JavaScript;Kotlin;Node.js;Objective-C;PHP;PowerShell;Pyth...
Recommendations:
- Your resume is moderately aligned (Likelihood: 60-80%). To improve:
- Consider adding these skills/technologies:
-   - Djangoexpressflaskjquerylaravelreactjsspringvuejsgoogle
-   - 3dangularangularjsaspnetaspnet
-   - Servermongodbmysqlpostgresql
-   - Cloud
-   - Core
- Emphasize projects or experiences using the job’s key technologies.
- Ensure your technical skills section is prominent and relevant.
--------------------------------------------------

Job #54 (Country: United States of America, Salary: $200,000.00)
Alignment Score: 0.65
Likelihood of Getting Job: 64.9%
Skills: C#;HT